# Install Dependecies

In [2]:
%%capture
%pip install -q "nltk>=3.9,<4" "spacy>=3.8,<4" "transformers>=5,<6"
%pip install matplotlib
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [3]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [4]:
import ipynbname
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import re
import time
from collections import Counter
from datasets import load_dataset
from pathlib import Path
from tqdm.std import tqdm
from transformers import AutoTokenizer

# Import datasets

In [6]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

# Preprocessing

In [7]:
LANGUAGES = ["ar", "ko", "te"]

In [8]:
# select languages
df_train = df_train[df_train["lang"].isin(LANGUAGES)].copy()
df_val = df_val[df_val["lang"].isin(LANGUAGES)].copy()

# 3 Week 37: Structured Span Prediction
* Convert the character-level answer offsets into BIO labels over context tokens. 
* Add automatic checks for at least the following cases: 
  - an answer at character 0, 
  - a multi-token answer, 
  - punctuation adjacent to an answer and an unanswerable example. 
* Document how subword pieces are handled if applicable. 
* Implement one question-conditioned sequence labeller for the group: the representation of the question must influence the predicted label for every context token. 
* Compare it with a simple span baseline. An empty-output baseline is sufficient. If you use lexical overlap, select context tokens using only overlap with the question (optionally after fixed preprocessing or translation), convert the best contiguous run to a span and never use gold answer text or offsets. The correct output for an unanswerable question is an empty span. Evaluate and analyse the models according to Section 1.


### 1. Convert the character-level answer offsets into BIO labels over context tokens

#### 1.1 Pin down the character span
The gold span is `answer_start` to `answer_start + len(answer)`, end exclusive. Project 1 already verified that slicing the context with these gives back the answer text. Also confirm what unanswerable rows look like, for example `answer_start == -1` and an empty answer, so they can be routed to the all-O case explicitly.

In [ ]:
def span_interval(df):
    df = df.copy()
    df["char_start"] = -1
    df["char_end"] = -1
    mask = df["answerable"]  # only apply to rows where answerable == True
    df.loc[mask, "char_start"] = df.loc[mask, "answer_start"]
    df.loc[mask, "char_end"] =  df.loc[mask, "answer_start"] + df.loc[mask, "answer"].str.len()
    return df

In [20]:
df_train = span_interval(df_train)
df_val = span_interval(df_val)
df_train.head(1)

5972
991


,question,context,lang,answerable,answer_start,answer,answer_inlang,char_start,char_end
4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None,21,27


#### 1.2 Tokenise with offsets

#### 1.3 Inspect how the offsets behave

#### 1.4 Define the token labelling rule

#### 1.5 Handle length

#### 1.6 Write the inverse function